# Example 14 — ROOT histogram efficiency and background

This example stores 2D efficiency and background maps as ROOT TH2 histograms, reloads them with uproot-backed helpers, and uses them in a Dalitz fit workflow.


In [ ]:
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import uproot

from dalitzplotfitter import (BackgroundCategory, DecayChannel, DecayModel, MultiBackgroundNLL, NonResonant, Parameter, RealImag, Resonance, enable_x64, histogram_background_from_root, histogram_efficiency_from_root, weighted_resample)
enable_x64()
channel=DecayChannel('B+',('K+','pi+','pi-'))
model=DecayModel(channel,[Resonance('Kstar892',(0,2),RealImag(1.0,0.0),mass=0.8958,width=0.0474,spin=1),Resonance('rho770',(1,2),RealImag(0.65,0.10),mass=0.7753,width=0.1491,spin=1),NonResonant(RealImag(-0.50,0.10))],normalization_method='square-dalitz',normalization_resolution=300,normalization_pair=(0,2))


## 1. Build synthetic TH2 maps in (s13,s23) and write them to ROOT


In [ ]:
pool=model.generate_phase_space(180000,seed=14001)
xlo,xhi=float(pool.s13.min()),float(pool.s13.max()); ylo,yhi=float(pool.s23.min()),float(pool.s23.max())
xedges=np.linspace(xlo,xhi,45); yedges=np.linspace(ylo,yhi,45)
xc=0.5*(xedges[:-1]+xedges[1:]); yc=0.5*(yedges[:-1]+yedges[1:]); X,Y=np.meshgrid(xc,yc,indexing='ij')
xn=(X-xlo)/(xhi-xlo); yn=(Y-ylo)/(yhi-ylo)
eff_values=np.clip(0.45+0.40*xn+0.10*np.cos(np.pi*yn),0.05,None)
bkg_values=0.30+1.20*xn+0.50*(1-yn)**2
root_path=Path('example14_maps.root')
with uproot.recreate(root_path) as f:
    f['efficiency_s13_s23']=(eff_values,xedges,yedges)
    f['background_s13_s23']=(bkg_values,xedges,yedges)
print(root_path.resolve())


## 2. Load ROOT TH2 maps directly as fitter models


In [ ]:
efficiency=histogram_efficiency_from_root(root_path,'efficiency_s13_s23',x_variable='s13',y_variable='s23')
background=histogram_background_from_root(root_path,'background_s13_s23',x_variable='s13',y_variable='s23')
fig,ax=plt.subplots(1,2,figsize=(12,4.8))
im0=ax[0].pcolormesh(np.asarray(efficiency.x_edges),np.asarray(efficiency.y_edges),np.asarray(efficiency.values).T,shading='auto'); fig.colorbar(im0,ax=ax[0]); ax[0].set_title('Efficiency from ROOT TH2')
im1=ax[1].pcolormesh(np.asarray(background.x_edges),np.asarray(background.y_edges),np.asarray(background.values).T,shading='auto'); fig.colorbar(im1,ax=ax[1]); ax[1].set_title('Background from ROOT TH2')
for a in ax: a.set_xlabel(r'$s_{13}$ [GeV$^2$]'); a.set_ylabel(r'$s_{23}$ [GeV$^2$]')
plt.show()


## 3. Generate pseudo-data with histogram efficiency and background


In [ ]:
eff_pool=efficiency(pool.as_dict()); bkg_pool=background(pool.as_dict())
N=25000; fs_true=0.80; ns=int(round(N*fs_true)); nb=N-ns
sig=weighted_resample(jax.random.key(14002),pool,pool.weights*eff_pool*model.intensity(pool.as_dict()),ns,replace=True)
bkg=weighted_resample(jax.random.key(14003),pool,pool.weights*bkg_pool,nb,replace=True)
from dalitzplotfitter.kinematics import PhaseSpaceSample
data=PhaseSpaceSample(s12=jnp.concatenate([sig.s12,bkg.s12]),s13=jnp.concatenate([sig.s13,bkg.s13]),s23=jnp.concatenate([sig.s23,bkg.s23]),weights=jnp.ones(N))
plt.figure(figsize=(7,5.5)); plt.hist2d(np.asarray(data.s13),np.asarray(data.s23),bins=70); plt.xlabel(r'$s_{13}$ [GeV$^2$]'); plt.ylabel(r'$s_{23}$ [GeV$^2$]'); plt.title('Toy data using ROOT histogram maps'); plt.colorbar(label='events'); plt.show()


## 4. Build the efficiency-corrected signal and normalized ROOT background


In [ ]:
norm=model.normalization_sample
signal_pdf=model.pdf(efficiency=efficiency)
bkg_norm=jnp.mean(norm.weights*background(norm.as_dict()))
f_sig=Parameter('signal_fraction',0.70,bounds=(0.05,0.99),step=0.01)
category=BackgroundCategory('root_hist_background',background(data.as_dict()),bkg_norm)
nll=MultiBackgroundNLL(signal_density=lambda values:signal_pdf(data.as_dict(),values),backgrounds=(category,),signal_fraction=f_sig)
from dalitzplotfitter import Minimizer
result=Minimizer(nll,(f_sig,),verbose=1).fit(start_values={'signal_fraction':0.70},simplex=True,ncall=10000)
print('generated signal fraction =',fs_true)
print('start signal fraction     = 0.70')
print('fitted signal fraction    =',float(result.values['signal_fraction']))
print('valid                     =',result.valid)


## 5. Compare accepted signal and background projections


In [ ]:
proj=model.generate_phase_space(120000,seed=14004)
sig_w=np.asarray(proj.weights*efficiency(proj.as_dict())*model.intensity(proj.as_dict()))
bkg_w=np.asarray(proj.weights*background(proj.as_dict()))
bins=np.linspace(xlo,xhi,70)
plt.figure(figsize=(7,5))
plt.hist(np.asarray(proj.s13),bins=bins,weights=sig_w,density=True,histtype='step',label='accepted signal')
plt.hist(np.asarray(proj.s13),bins=bins,weights=bkg_w,density=True,histtype='step',label='ROOT background')
plt.xlabel(r'$s_{13}$ [GeV$^2$]'); plt.ylabel('normalized density'); plt.legend(); plt.show()
